# NGC 1068 low-flux Case 1 versus Testagrossa-reference CPL

The left panel fits the thermal Case 1 spectrum at K/100 = 0.0031. The right panel reproduces the CPL spectrum from `NGC1068_Paper_Plots_NTH_Testagrossa.ipynb` at K/1000 = 0.00031 and index −1.85. Both now use independently fluctuated 300-seed ensembles.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display

UTILS_DIR = Path(
    "/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/"
    "continuum_fit/AGN/Fluctuate_True"
)
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

from agn_cosi_fit_utils import save_agn_fit_summary
from agn_ngc1068_fit import (
    build_sed_products,
    evaluate_flux_curves,
    fit_manifest_spectrum,
    plot_sed,
    plot_sed_pair,
)
%matplotlib inline


In [ ]:
BACKGROUND_PSEUDOCOUNT = None
SED_ENSEMBLE_RECOMPUTE = False
N_SED_BINS = 10

manifest_paths = {
    "low_flux_3m": Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_Case1_lowFlux100/NGC1068_Case1_lowFlux100_CPL_median_realization_3months.json"),
    "low_flux_24m": Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_Case1_lowFlux100/NGC1068_Case1_lowFlux100_CPL_median_realization_24months.json"),
    "testagrossa_3m": Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_TestagrossaReference/NGC1068_TestagrossaReference_CPL_median_realization_3months.json"),
    "testagrossa_24m": Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_TestagrossaReference/NGC1068_TestagrossaReference_CPL_median_realization_24months.json"),
}
FIT_SUMMARY_PATH = Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/Papers/AGN_Corona_EC_Overleaf/Fits/Fluctuate_True/NGC1068_Case1_lowFlux100_vs_Testagrossa_fit_summary_3_24Months.txt")
PLOT_DIR = Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/Papers/AGN_Corona_EC_Overleaf/Plots/Fluctuate_True")


In [ ]:
fits = {}
for label, manifest_path in manifest_paths.items():
    fits[label] = fit_manifest_spectrum(
        manifest_path,
        dataset_name=f"ngc1068_{label}",
        background_pseudocount=BACKGROUND_PSEUDOCOUNT,
    )
    print(label, "source TS =", f"{fits[label]['source_ts']:.3f}")
    display(fits[label]["likelihood"].results.get_data_frame())


In [ ]:
fit_summary_path = save_agn_fit_summary(
    output_path=FIT_SUMMARY_PATH,
    fit_results={label: bundle["likelihood"].results for label, bundle in fits.items()},
    injected_models={label: {"spectrum": bundle["injected_shape"]} for label, bundle in fits.items()},
    ts_values={label: bundle["source_ts"] for label, bundle in fits.items()},
    exposure_months={label: bundle["exposure_months"] for label, bundle in fits.items()},
    notes=(
        "Comparison of the K/100 thermal Case 1 spectrum with the K/1000, "
        "index=-1.85 Testagrossa-reference CPL. Both use fluctuated median files."
    ),
)
print(f"Saved: {fit_summary_path}")


In [ ]:
fit_low_24m = fits["low_flux_24m"]
fit_testagrossa_24m = fits["testagrossa_24m"]
curves_low = evaluate_flux_curves(fit_low_24m)
curves_testagrossa = evaluate_flux_curves(fit_testagrossa_24m)
sed_low = build_sed_products(
    fit_low_24m, n_sed_bins=N_SED_BINS,
    background_pseudocount=BACKGROUND_PSEUDOCOUNT,
    recompute_ensemble=SED_ENSEMBLE_RECOMPUTE,
    expected_seed_count=300,
)
sed_testagrossa = build_sed_products(
    fit_testagrossa_24m, n_sed_bins=N_SED_BINS,
    background_pseudocount=BACKGROUND_PSEUDOCOUNT,
    recompute_ensemble=SED_ENSEMBLE_RECOMPUTE,
    expected_seed_count=300,
)
display(sed_low["ensemble_summary"][["bin_index", "ts_median", "plot_role"]])
display(sed_testagrossa["ensemble_summary"][["bin_index", "ts_median", "plot_role"]])


## Plot A — representative median data sets

Each panel uses its selected realization’s own bin SED values and TS classification.


In [ ]:
representative_panels = [
    {
        "fit_bundle": fit_low_24m,
        "curves": curves_low,
        "sed_dataframe": sed_low["representative"],
        "title": "NGC 1068\nThermal CPL, K/100\n24-month",
        "color": "#D55E00",
    },
    {
        "fit_bundle": fit_testagrossa_24m,
        "curves": curves_testagrossa,
        "sed_dataframe": sed_testagrossa["representative"],
        "title": "NGC 1068\nTestagrossa-reference CPL, K/1000\n24-month",
        "color": "#2A8BC3",
    },
]
fig_representative, axes_representative = plot_sed_pair(
    representative_panels,
    save_path=PLOT_DIR / "NGC1068_Case1_lowFlux100_vs_Testagrossa_SED_RepresentativeMedianDataset.pdf",
)
plt.show()


## Plot B — 300-seed median SEDs

Each panel uses its independent 300-seed median SED values, intervals, and TS classification.


In [ ]:
ensemble_panels = [
    {**representative_panels[0], "sed_dataframe": sed_low["ensemble"]},
    {**representative_panels[1], "sed_dataframe": sed_testagrossa["ensemble"]},
]
fig_ensemble, axes_ensemble = plot_sed_pair(
    ensemble_panels,
    save_path=PLOT_DIR / "NGC1068_Case1_lowFlux100_vs_Testagrossa_SED_300SeedMedianSED.pdf",
)
plt.show()
